# Problem Session 5: Logistic Regression and Poisson Regression

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")

### Logistic Regression to predict Diabetes Status

The following dataset comes from [Kaggle](https://www.kaggle.com/datasets/ehababoelnaga/diabetes-dataset/data).

In [2]:
df_train = pd.read_csv('../../data/diabetes_train.csv')
df_test = pd.read_csv('../../data/diabetes_test.csv')
X_train = df_train.iloc[:,:-1]
y_train = df_train.iloc[:,-1]
X_test = df_test.iloc[:,:-1]
y_test = df_test.iloc[:,-1]
X_train


,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
2455,3,126,88,41,235,39.3,0.704,27
2456,4,123,62,0,0,32.0,0.226,35
2457,1,80,74,11,60,30.0,0.527,22
2458,1,96,64,27,87,33.2,0.289,21


Use seaborn pairplot to look at scatterplots of all pairs of features, with color determined by the target.

`BMI` and `Glucose` appear to be fairly good at seperating the classes.  They also have the advantage of being easily measured at home.

In [ ]:
plt.scatter(X_train[y_train == 0]['BMI'], X_train[y_train == 0]["Glucose"])
plt.scatter(X_train[y_train == 1]['BMI'], X_train[y_train == 1]["Glucose"])

plt.show()

It seems that the dataset contains some zero values for these features.  We don't want those contaminating our model, so we will redefine `X_train` and `y_train` to only include those observations where neither feature is equal to zero.

In [ ]:
y_train = 
X_train = 

In [ ]:
plt.scatter(X_train[y_train == 0]['BMI'], X_train[y_train == 0]["Glucose"])
plt.scatter(X_train[y_train == 1]['BMI'], X_train[y_train == 1]["Glucose"])

plt.show()

Compare a logistic regression model with no feature engineering to another logistic regression model where we first create quadratic polynomial features (leading to quadratic decision boundaries).  Use `penalty = None` for both.  For each model, keep track of cross validation roc_auc, log_loss, and accuracy means and standard deviations.

It looks like the two models perform almost identically. We should select the less complicated model. Just to get a feel for what they look like, train both models on the entire training set and look at their decision boundaries on the training set using the following code:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression


# 1) models
lr_linear = 

lr_poly2 =

# 2) fit on full training set


#### Rest is completed for you

# Palette (Okabe–Ito + neutral boundary)
COL_BG = "cividis"           # for contourf of P(y=1)
COL_BOUNDARY = "#3C3C3C"     # dark gray 0.5 contour
COL_NEG = "#0072B2"          # class 0: blue
COL_POS = "#D55E00"          # class 1: vermillion

def plot_boundary(ax, model, X, y, title):
    x1 = X.iloc[:,0].values
    x2 = X.iloc[:,1].values
    pad1 = 0.05 * (x1.max() - x1.min())
    pad2 = 0.05 * (x2.max() - x2.min())
    x1_min, x1_max = x1.min() - pad1, x1.max() + pad1
    x2_min, x2_max = x2.min() - pad2, x2.max() + pad2
    xx1, xx2 = np.meshgrid(
        np.linspace(x1_min, x1_max, 400),
        np.linspace(x2_min, x2_max, 400)
    )
    grid = pd.DataFrame(
    np.c_[xx1.ravel(), xx2.ravel()],
    columns=["Glucose", "BMI"]
    )
    proba = model.predict_proba(grid)[:, 1].reshape(xx1.shape)  

    # background probability field
    ax.contourf(xx1, xx2, proba, levels=np.linspace(0, 1, 21), cmap=COL_BG, alpha=0.85)
    # 0.5 decision boundary
    ax.contour(xx1, xx2, proba, levels=[0.5], linewidths=2.5, colors=COL_BOUNDARY)

    # class points with explicit colors
    y = np.asarray(y)
    ax.scatter(x1[y==0], x2[y==0], s=22, c=COL_NEG, edgecolors="white", linewidths=0.4, label="class 0")
    ax.scatter(x1[y==1], x2[y==1], s=22, c=COL_POS, edgecolors="white", linewidths=0.4, label="class 1")

    ax.set_xlabel("Glucose"); ax.set_ylabel("BMI"); ax.set_title(title)
    ax.legend(frameon=True, fancybox=False, edgecolor="0.8")


# 4) draw side-by-side
fig, axes = plt.subplots(1, 2, figsize=(10, 4.8), constrained_layout=True)
plot_boundary(axes[0], lr_linear, X_train, y_train, "Unregularized LR (2 features)")
plot_boundary(axes[1], lr_poly2, X_train, y_train, "Unregularized LR + degree-2 poly")
plt.show()

### Poisson Regression to predict number of insurance claims

Data description:  https://dutangc.github.io/CASdatasets/reference/freMTPL.html

In [ ]:
import pandas as pd
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.linear_model import PoissonRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error, mean_poisson_deviance

# 1) Data
df = fetch_openml(data_id=41214, as_frame=True).frame  # French motor TPL frequency

In [ ]:
# Look at df to get a feel for the data.

In [ ]:
# How many clients have 0 claims?  1 claim? 2?  What pandas dataframe method can help you answer this question in a one liner?

In [ ]:
y_count = df["ClaimNb"].astype(int)

expo = df["Exposure"].clip(lower=1e-6) # The period of exposure for a policy, in years.

X = df[["VehPower", "VehAge", "DrivAge", "VehBrand", "Region"]]
num_cols = ["VehPower", "VehAge", "DrivAge"]
cat_cols = ["VehBrand", "Region"]

# 2) Train/test split
X_tr, X_te, y_tr_count, y_te_count, exp_tr, exp_te = train_test_split(
    X, y_count, expo, test_size=0.3, random_state=42
)

We will fit a Poisson generalized linear model with a log link using scikit-learn’s PoissonRegressor. 

In lecture we discussed a Poisson regression model which requires our target to be a count in a given fixed unit of time.  In this case, however, different individuals have held their policies for different amounts of time (different amounts of "Exposure").  To meet the assumptions of the model, we define our target as the claim rate, defined as ClaimNb divided by Exposure, but then also weight by exposure.  The explanation for why this trick is necessary can be found here:

https://stats.stackexchange.com/questions/264071/how-is-a-poisson-rate-regression-equal-to-a-poisson-regression-with-correspondin?utm_source=chatgpt.com

In [ ]:
# 3) Target as RATE = count / exposure; weights = exposure
y_tr_rate = 
y_te_rate = 

In [ ]:
# 4) Preprocess + model


In [ ]:
# 5) Fit with sample_weight = exposure.  
# Note the double underscore convention for accessing kwargs of a component of a pipeline.  This is not super well documented.
model.fit(X_tr, y_tr_rate, poi__sample_weight=exp_tr)

In [ ]:
# 6) Predict rate, then counts = rate * exposure
pred_rate = 
pred_count = 

In [ ]:
# 7) Evaluation (counts)
rmse = root_mean_squared_error(y_te_count, pred_count)
mpd = mean_poisson_deviance(y_te_count, pred_count)
mean_ratio = y_te_count.mean() / pred_count.mean()

print(f"RMSE (counts): {rmse:.6f}")
print(f"Mean Poisson deviance: {mpd:.6f}")
print(f"Observed mean: {y_te_count.mean():.6f}")
print(f"Predicted mean: {pred_count.mean():.6f}")
print(f"Mean(obs)/Mean(pred): {mean_ratio:.6f}")

# 8) Sanity checks for dispersion on train (counts)
print("\nDispersion check (train):")
print("Mean(count):", y_tr_count.mean(), "Var(count):", y_tr_count.var(), "Var/Mean:", y_tr_count.var()/y_tr_count.mean())

--------------------------

This notebook was written for the Erdős Institute Data Science Boot Camp by Steven Gubkin.

Please refer to the license in this repo for information on redistribution.